# Notebook 02: Reproduce RQ2 tables

This notebook reproduces Tables 3 and 4 of the paper from
``../data/rq2_discovery.csv`` and ``../data/rq2_efficiency.csv``.
Table 3 covers the discovery dimension (bug counts, exclusive-bug
proportion p_excl with Wilson 95% CI, crash-input productivity).
Table 4 covers the efficiency dimension (per-second bug rate R_bug,
MTTFB, percentage reductions, early-warning advantage, median
advantage on shared bugs).


In [ ]:
import csv
import math
from pathlib import Path

DATA_DIR = Path("..") / "data"

with (DATA_DIR / "rq2_discovery.csv").open(newline="") as handle:
    discovery = list(csv.DictReader(handle))
with (DATA_DIR / "rq2_efficiency.csv").open(newline="") as handle:
    efficiency = list(csv.DictReader(handle))


## Reproduce Table 3 (discovery dimension)


In [ ]:
header = (f"{'Campaign':<10}{'Bugs(EM)':>10}{'Bugs(B)':>10}"
          f"{'Seeds(EM)':>11}{'Seeds(B)':>10}{'p_excl':>9}"
          f"{'Wilson95':>20}{'Prod(EM)':>10}{'Prod(B)':>10}")
print(header)
print("-" * len(header))
total_em, total_base = 0, 0
for row in discovery:
    bugs_em = int(row["bugs_em"])
    bugs_base = int(row["bugs_base"])
    total_em += bugs_em
    total_base += bugs_base
    ci = f"[{float(row['wilson_low']):.3f},{float(row['wilson_high']):.3f}]"
    print(f"{row['campaign']:<10}{bugs_em:>10}{bugs_base:>10}"
          f"{int(row['seeds_em']):>11}{int(row['seeds_base']):>10}"
          f"{float(row['p_excl']):>9.2f}{ci:>20}"
          f"{float(row['prod_crash_em']):>10.2f}{float(row['prod_crash_base']):>10.2f}")

print()
print(f"Total bugs EM:       {total_em}")
print(f"Total bugs baseline: {total_base}")
assert total_em == 8, f"Expected EM total 8, got {total_em}"
assert total_base == 4, f"Expected baseline total 4, got {total_base}"


## Verify Wilson 95% CIs from first principles


In [ ]:
def wilson_ci(k: int, n: int) -> tuple:
    z = 1.959963984540054
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

cases = [("C2", 1, 2), ("C3", 0, 2), ("picoc", 3, 4)]
by_camp = {row["campaign"]: row for row in discovery}
print(f"{'Campaign':<10}{'Stored':>22}{'Recomputed':>22}")
print("-" * 54)
for camp, k, n in cases:
    stored = f"[{float(by_camp[camp]['wilson_low']):.3f}, {float(by_camp[camp]['wilson_high']):.3f}]"
    rec_low, rec_high = wilson_ci(k, n)
    rec = f"[{rec_low:.3f}, {rec_high:.3f}]"
    print(f"{camp:<10}{stored:>22}{rec:>22}")


## Reproduce Table 4 (efficiency dimension)


In [ ]:
header = (f"{'Campaign':<10}{'R_bug(EM)':>11}{'R_bug(B)':>10}"
          f"{'MTTFB(EM)':>11}{'MTTFB(B)':>10}{'Red.%':>8}"
          f"{'Early':>10}{'MedAdv(s)':>11}")
print(header)
print("-" * len(header))
for row in efficiency:
    mttfb_em = float(row["mttfb_em_s"])
    mttfb_base = float(row["mttfb_base_s"])
    reduction = (mttfb_base - mttfb_em) / mttfb_base * 100
    early = f"{int(row['early_by_em_num'])}/{int(row['early_by_em_den'])}"
    print(f"{row['campaign']:<10}"
          f"{float(row['r_bug_em']):>11.2f}{float(row['r_bug_base']):>10.2f}"
          f"{mttfb_em:>10.1f}s{mttfb_base:>9.1f}s"
          f"{reduction:>7.1f}%{early:>10}{float(row['median_advantage_s']):>11.1f}")


## Aggregate detection-rate ratio

Two views of the EM-vs-baseline detection-rate ratio across campaigns:
the per-campaign ratios and their median, and the study-level
aggregate computed from mean R_bug values.


In [ ]:
ratios = []
for row in efficiency:
    r_em = float(row["r_bug_em"])
    r_base = float(row["r_bug_base"])
    ratios.append((row["campaign"], r_em / r_base))

for camp, r in ratios:
    print(f"  {camp}: {r:.2f}x")

values = sorted(r for _, r in ratios)
median = values[len(values) // 2]
mean_per_campaign = sum(values) / len(values)

r_em_values = [float(row['r_bug_em']) for row in efficiency]
r_base_values = [float(row['r_bug_base']) for row in efficiency]
aggregate = (sum(r_em_values) / len(r_em_values)) / (sum(r_base_values) / len(r_base_values))

print()
print(f"Median per-campaign ratio:    {median:.2f}x")
print(f"Arithmetic mean per-campaign: {mean_per_campaign:.2f}x")
print(f"Study-level aggregate ratio:  {aggregate:.2f}x")


Expected: per-campaign ratios 1.95, 1.00, 4.05; median 1.95x;
mean per-campaign approximately 2.33x; aggregate 1.90x.

At n = 3 campaigns the median is more representative than the aggregate.
The aggregate is dominated by picoc, which has the highest per-second
bug rate. The median is the value reported as the headline summary in
the paper.
